# TalentDesk, Module 3 Section 2 Lab (Exercise): Commands, Skills, and Execution Modes

A hands-on exercise on packaging repeated workflows and matching the execution mode to the task. It
combines the two M3S2 skills: turning a workflow into a **slash command** or **skill** with frontmatter,
`allowed-tools`, and `context: fork` (Lab 1), and choosing **plan mode**, **direct execution**, or the
**Explore** subagent by the shape of the task (Lab 2). You fill in four short `TODO` blocks over a
TalentDesk sandbox project; everything else is provided. All four are testable offline, and live
**Claude Agent SDK** cells invoke a command and show plan versus direct execution. Runs **Sonnet**
(`claude-sonnet-4-6`).

## The real-world scenario

Every TalentDesk teammate types out the same candidate-screening checklist by hand, and every offer
change gets a manual audit. Both are repeatable, so both belong in the repo as tools: a `/screen-review`
**command** any teammate gets on clone, and an `offer-audit` **skill** that runs in its own context and
can only read, never write.

Then there is the work itself. Reshaping the screening pipeline across several modules deserves a
reviewed **plan** before any code changes; a one-line change to the bar just needs **direct execution**;
and understanding the pipeline first is a job for **Explore**, which investigates in isolation and
returns a summary.

The question this lab answers: **how do you package a workflow as a command or skill and keep it
least-privileged, and when do you plan, execute directly, or delegate discovery to Explore?**

## Objectives

- Create a project **slash command** with frontmatter and `$ARGUMENTS`, and understand a **skill** with
  `context: fork`.
- Enforce **allowed-tools** so a workflow can only do what its role needs.
- Choose **plan** vs **direct** from the task shape, and measure the context **Explore** (or a forked
  skill) saves.

## The outcome you should reach

By the end you will have:

- argument substitution that fills `$ARGUMENTS` and `$1`/`$2` in a command body;
- an `allowed-tools` check that permits Read and Grep but blocks Write;
- a mode classifier that recommends direct for a small fix and plan for a multi-file refactor;
- and a savings calculation showing an isolated summary costs a fraction of reading everything inline.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live cells need a
real key and Node.js 18+.

## How to run

Run top to bottom. Building the sandbox and the pure-Python cells run anywhere. The live cells invoke a
command and run plan versus direct execution through Claude, so paste a real key into **Setup 2/3** and
re-run from the top; **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses the YAML frontmatter; the Agent SDK drives
the live cells and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, `run_async()`, and a `snapshot()` helper
that hashes every file so the live cells can prove which mode changed the repo.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, an async runner, a snapshot =====
import os                                       # filesystem paths for the sandbox project
import re                                       # text substitution and frontmatter matching
import sys                                       # detect Windows (special event loop)
import yaml                                      # parse the YAML frontmatter
import hashlib                                  # hash files to detect changes
import textwrap                                  # keeps the embedded file bodies readable
from pathlib import Path                          # walk the repo
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cells will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

def snapshot(root):                              # map every file -> its content hash
    return {str(p.relative_to(root)): hashlib.md5(p.read_bytes()).hexdigest()
            for p in sorted(Path(root).rglob("*")) if p.is_file()}

print("live model calls:", "ON" if RUN_LIVE else "OFF (sandbox cells run offline)")

**This cell:** writes a small **sandbox project** (provided): a `/screen-review` command, an
`offer-audit` skill, and a few TalentDesk modules for them to act on. We use a sandbox folder so nothing
touches your real `~/.claude`. The command and skill are plain text, exactly what you would commit.

In [ ]:
# ===== SETUP 3/3 - create the sandbox project (command + skill + modules) =====
PROJECT = os.path.join(os.getcwd(), "talentdesk_project")

FILES = {
    ".claude/commands/screen-review.md": textwrap.dedent("""\
        ---
        description: Review a screening module against our checklist
        argument-hint: [file-path]
        allowed-tools: Read, Grep
        ---
        Review $ARGUMENTS against the TalentDesk screening checklist:
        1. Every decision checks the bar (MIN_YEARS).
        2. No candidate PII is logged in plain text.
        3. Public functions have a docstring.
        Report findings as a short bulleted list. Do not edit anything.
        """),
    ".claude/skills/offer-audit/SKILL.md": textwrap.dedent("""\
        ---
        name: offer-audit
        description: Audit offer logic for band checks and hardcoded secrets. Use before shipping offer changes.
        context: fork
        allowed-tools: Read, Grep
        argument-hint: [directory]
        ---
        Audit the offer logic under $ARGUMENTS.
        1. Grep for offer functions.
        2. Confirm each checks the salary band.
        3. Flag any hardcoded secret.
        Return only a short summary of findings.
        """),
    "talentdesk/screening.py": textwrap.dedent("""\
        MIN_YEARS = 3

        def meets_bar(candidate):
            return candidate["years"] >= MIN_YEARS

        def screen_candidate(candidate):
            return "advance" if meets_bar(candidate) else "reject"
        """),
    "talentdesk/candidates.py": 'CANDIDATES = {"C1": {"years": 5}, "C2": {"years": 1}}\n',
    "talentdesk/scheduling.py": 'def get_slot(candidate_id):\n    return "slot for " + candidate_id\n',
}
for rel, content in FILES.items():
    path = os.path.join(PROJECT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created project at", PROJECT, "with", len(FILES), "files")

### Commands and skills, side by side

A **slash command** is a Markdown file in `.claude/commands/` (project, shared via Git) or
`~/.claude/commands/` (personal); the filename is the command name (`screen-review.md` becomes
`/screen-review`). A **skill** is a folder `.claude/skills/<name>/SKILL.md` that does everything a
command does plus richer frontmatter and supporting files. Both use YAML frontmatter (`description`,
`allowed-tools`, `argument-hint`, `model`); skills add `context: fork` to run in an isolated subagent.
`$ARGUMENTS` is the whole trailing string; `$1`, `$2` are positional.

---

### 🎯 Part A - package, parameterize, restrict

**This cell:** a parser that splits a command or skill file into its **frontmatter** and **body**
(provided), exactly what Claude Code does when it loads the file.

In [ ]:
# ===== parse a command/skill file into (frontmatter, body) (provided) =====
def parse_config(rel):                             # rel path under PROJECT -> (dict, body_text)
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
    if not m:
        return {}, text
    return (yaml.safe_load(m.group(1)) or {}), m.group(2)

front, body = parse_config(".claude/commands/screen-review.md")
print("frontmatter:", front)
print("body starts:", body.splitlines()[0])

**TODO 1 (about 4 minutes).** Complete `apply_args()`, the argument substitution. Replace
`$ARGUMENTS` with the whole trailing string, and `$1`, `$2`, ... with the positional words (one-indexed).
This is what turns `/screen-review talentdesk/screening.py` into the final prompt.

In [ ]:
# ===== TODO 1 - fill $ARGUMENTS and $1, $2 in the body =====
def apply_args(body, argstr):                      # body + "a b c" -> body with args substituted
    parts = argstr.split()                          # positional words
    # 👉 TODO 1a: filled = body.replace("$ARGUMENTS", argstr)   # the whole trailing string
    filled = body
    # 👉 TODO 1b: for i, word in enumerate(parts, 1): filled = filled.replace(f"${i}", word)
    return filled

print(apply_args(body, "talentdesk/screening.py").splitlines()[0])

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 1 =====
assert apply_args("all: $ARGUMENTS", "C1 C2") == "all: C1 C2", "$ARGUMENTS is the whole string"
assert apply_args("check $1 then $2", "C1 C2") == "check C1 then C2", "positional args fill in"
assert "talentdesk/screening.py" in apply_args(body, "talentdesk/screening.py"), "the file name lands in the body"
print("TODO 1 checks passed")

**TODO 2 (about 4 minutes).** Complete `allowed_set()`, which reads the `allowed-tools` frontmatter
into a set of base tool names. The spec is a comma-separated string like `"Read, Grep"` or
`"Bash(git add:*), Read"`; take the base name before any `(`. This is how a command or skill is kept
least-privileged.

In [ ]:
# ===== TODO 2 - enforce allowed-tools =====
def allowed_set(front):                            # frontmatter -> set of permitted base tool names
    spec = front.get("allowed-tools", "")           # e.g. "Read, Grep" or "Bash(git add:*)"
    # 👉 TODO 2: split spec on ",", strip each, take the part before "(", drop blanks, return a set
    #    names = [t.strip().split("(")[0] for t in spec.split(",") if t.strip()]
    #    return set(names)
    return set()

allowed = allowed_set(front)
print("allowed:", allowed)
for tool in ["Read", "Grep", "Write"]:
    print(f"  {tool:5} -> {'allowed' if tool in allowed else 'BLOCKED'}")

**Self-check (offline).** Read and Grep are permitted; Write is blocked; a `Bash(...)` spec reduces
to `Bash`.

In [ ]:
# ===== self-check for TODO 2 =====
assert allowed_set({"allowed-tools": "Read, Grep"}) == {"Read", "Grep"}
assert "Write" not in allowed_set({"allowed-tools": "Read, Grep"}), "Write must be blocked"
assert allowed_set({"allowed-tools": "Bash(git add:*), Read"}) == {"Bash", "Read"}, "take the base name"
assert allowed_set({}) == set(), "no allowed-tools -> empty set"
print("TODO 2 checks passed")

**This cell:** what `context: fork` buys you (provided). A forked skill runs in an isolated
subagent: it may read and search many files (verbose work), but only a short **summary** returns to your
main conversation. You will quantify this saving with your own function in Part B.

In [ ]:
# ===== the offer-audit skill is forked (provided) =====
skill_front, skill_body = parse_config(".claude/skills/offer-audit/SKILL.md")
print("skill:", skill_front.get("name"), "| context:", skill_front.get("context"),
      "| tools:", allowed_set(skill_front))
print("is isolated (context: fork)?", skill_front.get("context") == "fork")

---

### 🎯 Part B - match the mode to the task

**Direct execution** for a small, clear change. **Plan mode** for multi-file or architectural work: a
read-only gate that proposes a plan and waits for approval before touching anything. **Explore** for
discovery: read-only, runs in isolation, and returns a summary so raw file contents never enter your main
context. The strong pattern chains them: Explore to understand, plan to design, execute to implement.

**TODO 3 (about 5 minutes).** Complete `choose_mode()`, the classifier. Recommend **plan** when the
task touches three or more files or contains a scope signal (`refactor`, `redesign`, `architecture`,
`migrate`, `across`, `multiple`, `several`, `restructure`, `split`); otherwise recommend **direct**.

In [ ]:
# ===== TODO 3 - choose plan vs direct from the task shape =====
SIGNALS = ["refactor", "redesign", "architecture", "migrate", "across",
           "multiple", "several", "restructure", "split"]

def choose_mode(task, files_touched=1):            # task text + rough file count -> (mode, why)
    t = task.lower()
    hits = [w for w in SIGNALS if w in t]
    # 👉 TODO 3: if files_touched >= 3 or hits: return ("plan", f"multi-file or architectural ({hits or files_touched})")
    #            otherwise return ("direct", "small, well-scoped change")
    return ("direct", "small, well-scoped change")

for task, n in [("Fix the typo in the reject message", 1),
                ("Refactor the screening logic across candidates, screening, and scheduling", 3),
                ("Change MIN_YEARS from 3 to 5", 1),
                ("Redesign the pipeline to support multi-stage screening", 2)]:
    mode, why = choose_mode(task, n)
    print(f"  [{mode:6}] {task}  ({why})")

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 3 =====
assert choose_mode("Fix the typo in the reject message", 1)[0] == "direct"
assert choose_mode("Change MIN_YEARS from 3 to 5", 1)[0] == "direct"
assert choose_mode("Refactor across three modules", 3)[0] == "plan", "3+ files -> plan"
assert choose_mode("Redesign the pipeline", 2)[0] == "plan", "a scope signal -> plan"
print("TODO 3 checks passed")

**TODO 4 (about 4 minutes).** Complete `context_savings()`, which quantifies what an isolated pass
(Explore, or a forked skill) saves. Given the bytes that would be read inline and the summary text that
comes back, approximate tokens as characters over four and return `(inline_tokens, summary_tokens,
saved_fraction)` where `saved_fraction = 1 - summary_tokens / inline_tokens`.

In [ ]:
# ===== TODO 4 - Explore / fork savings: a summary vs reading everything inline =====
def context_savings(inline_bytes, summary):        # -> (inline_tokens, summary_tokens, saved_fraction)
    # 👉 TODO 4a: inline_tokens = inline_bytes // 4
    # 👉 TODO 4b: summary_tokens = len(summary) // 4
    # 👉 TODO 4c: return inline_tokens, summary_tokens, 1 - summary_tokens / inline_tokens
    return 0, 0, 0.0

repo_bytes = sum(len(p.read_text()) for p in Path(PROJECT).rglob("*.py"))
summary = ("Screening path: screen_candidate calls meets_bar, which checks MIN_YEARS (3). "
           "candidates.py holds the data; scheduling.py is unrelated.")
inline_t, summ_t, saved = context_savings(repo_bytes, summary)
print(f"read everything inline: {inline_t} tokens | Explore summary: {summ_t} tokens | saved {saved:.0%}")

**Self-check (offline).** A summary should cost far less than reading everything inline, and the
same function measures the `context: fork` skill's saving.

In [ ]:
# ===== self-check for TODO 4 =====
inline_t, summ_t, saved = context_savings(repo_bytes, summary)
assert inline_t > summ_t > 0, "the summary should be smaller but non-empty"
assert 0 < saved < 1, "saved should be a fraction"
# reuse it for the forked skill's verbose work vs its returned summary
fork_saved = context_savings(len("verbose audit work " * 200), "1 offer path; band check present; no secrets.")[2]
assert fork_saved > 0.5, "a forked skill returns only a summary, saving most of the tokens"
print(f"TODO 4 checks passed: Explore saved {saved:.0%}, forked skill saved {fork_saved:.0%}")

**This cell:** the live command invocation (provided). It points the Agent SDK at the sandbox with
`cwd=PROJECT` and `setting_sources=["project"]` so `.claude/commands/` loads, then invokes
`/screen-review`. With only Read, Grep, and Glob allowed, the review stays read-only. Offline it prints
the expected outcome.

In [ ]:
# ===== live: invoke the project slash command =====
try:
    from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock
    SDK_OK = True
    CMD_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT, setting_sources=["project"],
                                  allowed_tools=["Read", "Grep", "Glob"])
    async def run_cmd(prompt, opts):
        async for m in query(prompt=prompt, options=opts):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock): print("  ->", b.name)
                    elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])
    print("command options ready (cwd =", PROJECT, ")")
except Exception:
    SDK_OK = False
    print("Agent SDK not available offline; the sandbox logic above was tested with Python.")

if RUN_LIVE and SDK_OK:
    run_async(lambda: run_cmd("/screen-review talentdesk/screening.py", CMD_OPTS))
else:
    print("[offline] expected: Claude loads .claude/commands/screen-review.md and reviews the file read-only.")

**This cell:** the live plan-versus-direct demonstration (provided). Plan mode
(`permission_mode="plan"`) is a read-only gate: it proposes and changes nothing, proven by the snapshot.
Direct execution (`permission_mode="acceptEdits"`) makes the small change. Offline it prints the expected
outcomes.

In [ ]:
# ===== live: plan mode proposes; direct execution edits =====
if RUN_LIVE and SDK_OK:
    PLAN_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT, permission_mode="plan",
                                   allowed_tools=["Read", "Grep", "Glob"])
    EDIT_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT, permission_mode="acceptEdits",
                                   allowed_tools=["Read", "Grep", "Edit"])
    before = snapshot(PROJECT)
    run_async(lambda: run_cmd("Refactor the screening logic across candidates.py, screening.py, and scheduling.py.", PLAN_OPTS))
    changed = [f for f in before if before[f] != snapshot(PROJECT).get(f)]
    print("files changed by plan mode:", changed, "(expected: none)\n")

    before = snapshot(PROJECT)
    run_async(lambda: run_cmd("In talentdesk/screening.py change MIN_YEARS from 3 to 5. Read then Edit.", EDIT_OPTS))
    changed = [f for f in before if before[f] != snapshot(PROJECT).get(f)]
    print("files changed by direct execution:", changed, "(expected: talentdesk/screening.py)")
else:
    print("[offline] expected: plan mode proposes a refactor and edits NO files (snapshot unchanged);")
    print("          direct execution Reads then Edits screening.py (snapshot shows it changed).")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| paste the same checklist prompt every time | put it in `.claude/commands/` as `/screen-review` |
| give an audit skill full tool access | set `allowed-tools` to the minimum (Read, Grep) |
| let a verbose skill flood your context | add `context: fork` to isolate it |
| plan mode for a one-line fix | use direct execution; planning is overhead here |
| execute a multi-file refactor blind | use plan mode, review the plan, then execute |
| read many files into your main chat to explore | delegate to Explore; take the summary |

**Lesson:** a **command** is a shared prompt template; a **skill** is the same idea with richer
frontmatter and its own folder, and frontmatter is where the control lives (`allowed-tools` restricts,
`context: fork` isolates). And there is no single right execution mode, only a right match: **direct**
for small clear changes, **plan** for multi-file or architectural work where being wrong is costly, and
**Explore** to investigate without draining your context. The strongest workflow chains them: Explore to
understand, plan to design, execute to implement.

---

## Recap - packaging and execution modes

| Piece | Where / when | What it does |
|---|---|---|
| slash command | `.claude/commands/*.md` | shared prompt template, `$ARGUMENTS` (Lab 1) |
| skill | `.claude/skills/<name>/SKILL.md` | command plus frontmatter and files (Lab 1) |
| `allowed-tools` | frontmatter | restrict tools to least privilege (Lab 1) |
| `context: fork` | skill frontmatter | isolate execution, return a summary (Lab 1) |
| Direct | small, clear, single change | edits immediately (Lab 2) |
| Plan | many files or architectural | proposes, waits for approval, edits nothing yet (Lab 2) |
| Explore | you must understand the code first | reads in isolation, returns a summary (Lab 2) |

**Try it next:** add a `$1`/`$2` positional command and invoke it with two arguments. Then give plan mode
a multi-file task and confirm the snapshot stays unchanged until you approve.